In [10]:
import torch
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import os
import numpy as np

In [11]:
# 导入融合网络
from network.net_fusion import DualEncoderFusionAutoEncoder

In [12]:
def load_and_preprocess_image(image_path):
    """加载并预处理单张图像"""
    # 打开图像并转换为灰度图
    image = Image.open(image_path).convert('L')

    # 定义预处理步骤
    transform = transforms.Compose([
        transforms.ToTensor(),  # 转换为tensor并归一化到[0,1]
    ])

    # 应用预处理
    tensor_image = transform(image)

    # 添加batch维度 (1, 1, H, W)
    tensor_image = tensor_image.unsqueeze(0)

    return tensor_image, image


def load_and_preprocess_color_image(image_path):
    """加载并预处理彩色图像"""
    # 打开图像并转换为RGB
    image = Image.open(image_path).convert('RGB')

    # 转换为LAB颜色空间
    lab_image = image.convert('LAB')

    # 分离L通道(亮度)和AB通道(色彩)
    l_channel, a_channel, b_channel = lab_image.split()

    # 对L通道进行预处理
    transform = transforms.Compose([
        transforms.ToTensor(),  # 转换为tensor并归一化到[0,1]
    ])

    # 应用预处理到L通道
    l_tensor = transform(l_channel)

    # 添加batch维度 (1, 1, H, W)
    l_tensor = l_tensor.unsqueeze(0)

    return l_tensor, image, lab_image


def convert_gray_to_color(gray_tensor, original_lab):
    """将处理后的灰度图像与原始色彩信息结合生成彩色图像"""
    # 将处理后的灰度图像转换为PIL图像
    gray_np = gray_tensor.squeeze(0).squeeze(0).cpu().numpy()
    gray_pil = Image.fromarray((gray_np * 255).astype(np.uint8), mode='L')

    # 获取原始图像的色彩通道
    orig_l, orig_a, orig_b = original_lab.split()

    # 将处理后的亮度与原始色彩结合
    reconstructed_lab = Image.merge('LAB', (gray_pil, orig_a, orig_b))

    # 转换回RGB
    reconstructed_rgb = reconstructed_lab.convert('RGB')

    return reconstructed_rgb


def post_process_image(tensor_image, method='clip', clip_min=0.0, clip_max=1.0, gamma=1.0):
    """
    对图像张量进行后处理以防止过曝

    Args:
        tensor_image: 输入的图像张量
        method: 处理方法 ('clip', 'normalize', 'gamma')
        clip_min: 裁剪最小值
        clip_max: 裁剪最大值
        gamma: 伽马校正系数

    Returns:
        处理后的图像张量
    """
    if method == 'clip':
        # 简单裁剪到[0,1]范围
        processed = torch.clamp(tensor_image, clip_min, clip_max)
    elif method == 'normalize':
        # 归一化到[0,1]范围
        min_val = tensor_image.min()
        max_val = tensor_image.max()
        processed = (tensor_image - min_val) / (max_val - min_val + 1e-8)
    elif method == 'gamma':
        # 伽马校正
        clipped = torch.clamp(tensor_image, clip_min, clip_max)
        processed = torch.pow(clipped, gamma)
    else:
        processed = tensor_image

    return processed



In [13]:


def save_comparison_fusion(ir_image, vi_image, fused_image, save_path):
    """保存红外、可见光和融合图像的对比"""
    # 转换为numpy数组并移除batch维度
    ir_np = ir_image.squeeze(0).squeeze(0).cpu().numpy()
    vi_np = vi_image.squeeze(0).squeeze(0).cpu().numpy()
    fused_np = fused_image.squeeze(0).squeeze(0).cpu().numpy()

    # 创建对比图
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 显示红外图像
    axes[0].imshow(ir_np, cmap='gray')
    axes[0].set_title('Infrared Image')
    axes[0].axis('off')

    # 显示可见光图像
    axes[1].imshow(vi_np, cmap='gray')
    axes[1].set_title('Visible Image')
    axes[1].axis('off')

    # 显示融合图像
    axes[2].imshow(fused_np, cmap='gray')
    axes[2].set_title('Fused Image')
    axes[2].axis('off')

    # 保存图像
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"融合对比图像已保存到: {save_path}")


In [14]:
def save_color_fusion_comparison(ir_rgb, vi_rgb, fused_rgb, save_path):
    """保存红外、可见光和融合彩色图像的对比"""
    # 创建对比图
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 显示红外图像
    axes[0].imshow(ir_rgb)
    axes[0].set_title('Infrared Image')
    axes[0].axis('off')

    # 显示可见光图像
    axes[1].imshow(vi_rgb)
    axes[1].set_title('Visible Image')
    axes[1].axis('off')

    # 显示融合图像
    axes[2].imshow(fused_rgb)
    axes[2].set_title('Fused Image')
    axes[2].axis('off')

    # 保存图像
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"彩色融合对比图像已保存到: {save_path}")

In [15]:
def predict_fusion(ir_model_path, vi_model_path, fusion_model_path, ir_image_path, vi_image_path, output_path):
    """使用训练好的双编码器融合网络对红外和可见光图像进行融合预测"""
    # 检查CUDA是否可用
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 创建双编码器融合自编码器模型
    model = DualEncoderFusionAutoEncoder(
        in_channels=1,
        out_channels=1,
        en_out_conv=32,
        dense_Layer_out=64,
        dense_layers=3,
        dense_out=128,
        kernel_size=3,
        debug=False
    )

    # 加载训练好的模型权重
    # 注意：这里我们需要分别加载红外编码器、可见光编码器和解码器的权重
    # 假设模型权重已经按照特定方式组织，或者我们可以从单独的文件中加载

    # 加载整个模型权重（如果有的话）
    if os.path.exists(fusion_model_path):
        model.load_state_dict(torch.load(fusion_model_path, map_location=device))
    else:
        print("警告：未找到完整的融合模型权重文件")

    model.to(device)
    model.eval()

    # 加载并预处理红外和可见光图像
    ir_tensor_image, _ = load_and_preprocess_image(ir_image_path)
    vi_tensor_image, _ = load_and_preprocess_image(vi_image_path)

    ir_tensor_image = ir_tensor_image.to(device)
    vi_tensor_image = vi_tensor_image.to(device)

    print(f"红外图像尺寸: {ir_tensor_image.shape}")
    print(f"可见光图像尺寸: {vi_tensor_image.shape}")

    # 使用模型进行预测
    with torch.no_grad():
        fused_image = model(ir_tensor_image, vi_tensor_image)

    print(f"融合图像尺寸: {fused_image.shape}")

    # 保存对比结果
    save_comparison_fusion(ir_tensor_image, vi_tensor_image, fused_image, output_path)

    return ir_tensor_image, vi_tensor_image, fused_image

In [16]:
def predict_fusion_with_separate_models(ir_encoder_path, vi_encoder_path, decoder_path, fusion_model_path,
                                       ir_image_path, vi_image_path, output_path):
    """使用训练好的分离模型进行红外和可见光图像融合预测"""
    # 检查CUDA是否可用
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 创建双编码器融合自编码器模型
    model = DualEncoderFusionAutoEncoder(
        in_channels=1,
        out_channels=1,
        en_out_conv=32,
        dense_Layer_out=64,
        dense_layers=3,
        dense_out=128,
        kernel_size=3,
        debug=False
    )

    # 加载训练好的各部分权重
    # 加载红外编码器权重
    if os.path.exists(ir_encoder_path):
        model.ir_encoder.load_state_dict(torch.load(ir_encoder_path, map_location=device))
        print("已加载红外编码器权重")
    else:
        print("警告：未找到红外编码器权重文件")

    # 加载可见光编码器权重
    if os.path.exists(vi_encoder_path):
        model.vi_encoder.load_state_dict(torch.load(vi_encoder_path, map_location=device))
        print("已加载可见光编码器权重")
    else:
        print("警告：未找到可见光编码器权重文件")

    # 加载解码器权重
    if os.path.exists(decoder_path):
        model.decoder.load_state_dict(torch.load(decoder_path, map_location=device))
        print("已加载解码器权重")
    else:
        print("警告：未找到解码器权重文件")

    # 如果有融合网络的单独权重文件，也加载它
    if os.path.exists(fusion_model_path) and fusion_model_path != decoder_path and fusion_model_path != ir_encoder_path and fusion_model_path != vi_encoder_path:
        # 这种情况下，我们假设fusion_model_path包含了整个融合模型的权重
        model.load_state_dict(torch.load(fusion_model_path, map_location=device))
        print("已加载融合模型权重")

    model.to(device)
    model.eval()

    # 加载并预处理红外和可见光图像
    ir_tensor_image, _ = load_and_preprocess_image(ir_image_path)
    vi_tensor_image, _ = load_and_preprocess_image(vi_image_path)

    ir_tensor_image = ir_tensor_image.to(device)
    vi_tensor_image = vi_tensor_image.to(device)

    print(f"红外图像尺寸: {ir_tensor_image.shape}")
    print(f"可见光图像尺寸: {vi_tensor_image.shape}")

    # 使用模型进行预测
    with torch.no_grad():
        fused_image = model(ir_tensor_image, vi_tensor_image)

    print(f"融合图像尺寸: {fused_image.shape}")

    # 保存对比结果
    save_comparison_fusion(ir_tensor_image, vi_tensor_image, fused_image, output_path)

    return ir_tensor_image, vi_tensor_image, fused_image



In [17]:

def predict_color_fusion_with_separate_models(ir_encoder_path, vi_encoder_path, decoder_path, fusion_model_path,
                                             ir_image_path, vi_image_path, output_path):
    """使用训练好的分离模型进行红外和可见光彩色图像融合预测"""
    # 检查CUDA是否可用
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 创建双编码器融合自编码器模型
    model = DualEncoderFusionAutoEncoder(
        in_channels=1,
        out_channels=1,
        en_out_conv=32,
        dense_Layer_out=64,
        dense_layers=3,
        dense_out=128,
        kernel_size=3,
        debug=False
    )

    # 加载训练好的各部分权重
    # 加载红外编码器权重
    if os.path.exists(ir_encoder_path):
        model.ir_encoder.load_state_dict(torch.load(ir_encoder_path, map_location=device))
        print("已加载红外编码器权重")
    else:
        print("警告：未找到红外编码器权重文件")

    # 加载可见光编码器权重
    if os.path.exists(vi_encoder_path):
        model.vi_encoder.load_state_dict(torch.load(vi_encoder_path, map_location=device))
        print("已加载可见光编码器权重")
    else:
        print("警告：未找到可见光编码器权重文件")

    # 加载解码器权重
    if os.path.exists(decoder_path):
        model.decoder.load_state_dict(torch.load(decoder_path, map_location=device))
        print("已加载解码器权重")
    else:
        print("警告：未找到解码器权重文件")

    # 如果有融合网络的单独权重文件，也加载它
    if os.path.exists(fusion_model_path) and fusion_model_path != decoder_path and fusion_model_path != ir_encoder_path and fusion_model_path != vi_encoder_path:
        # 这种情况下，我们假设fusion_model_path包含了整个融合模型的权重
        model.load_state_dict(torch.load(fusion_model_path, map_location=device))
        print("已加载融合模型权重")

    model.to(device)
    model.eval()

    # 加载并预处理红外和可见光彩色图像
    ir_l_tensor, ir_rgb_image, ir_lab_image = load_and_preprocess_color_image(ir_image_path)
    vi_l_tensor, vi_rgb_image, vi_lab_image = load_and_preprocess_color_image(vi_image_path)

    ir_l_tensor = ir_l_tensor.to(device)
    vi_l_tensor = vi_l_tensor.to(device)

    print(f"红外图像尺寸: {ir_l_tensor.shape}")
    print(f"可见光图像尺寸: {vi_l_tensor.shape}")

    # 使用模型进行预测
    with torch.no_grad():
        fused_l_tensor = model(ir_l_tensor, vi_l_tensor)

    print(f"融合图像尺寸: {fused_l_tensor.shape}")

    # 将处理后的L通道与原始色彩信息结合生成彩色图像
    # 这里我们使用可见光图像的色彩信息作为参考
    fused_rgb_image = convert_gray_to_color(fused_l_tensor, vi_lab_image)

    # 保存彩色对比结果
    save_color_fusion_comparison(ir_rgb_image, vi_rgb_image, fused_rgb_image, output_path)

    # 同时保存重建的彩色图像
    output_dir = os.path.dirname(output_path)
    base_name = os.path.splitext(os.path.basename(output_path))[0]
    reconstructed_image_path = os.path.join(output_dir, f"{base_name}_color_fused.png")
    fused_rgb_image.save(reconstructed_image_path)
    print(f"重建彩色图像已保存到: {reconstructed_image_path}")

    return ir_rgb_image, vi_rgb_image, fused_rgb_image

In [18]:
if __name__ == "__main__":
    # 模型路径（请根据实际情况修改）
    ir_encoder_path = r"../weights/encoder_final.pth"  # 红外编码器权重路径
    vi_encoder_path = r"../weights/encoder_final.pth"  # 可见光编码器权重路径（这里假设使用相同的编码器）
    decoder_path = r"../weights/decoder_final.pth"     # 解码器权重路径
    fusion_model_path = ""                         # 如果有单独的融合模型权重文件，请指定路径

    # 测试图像路径（请根据实际情况修改）
    ir_image_path = r"../image/testNet/250256_ir.jpg"      # 红外图像路径
    vi_image_path = r"../image/testNet/250256_vi.jpg"       # 可见光图像路径

    # 输出图像路径
    output_path = r"../output/fusion/fusion_comparison.png"
    color_output_path = r"../output/fusion/color_fusion_comparison.png"

    # 确保输出目录存在
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # 进行预测
    try:
        print("使用分离的编码器和解码器模型进行红外与可见光图像融合...")
        ir_tensor, vi_tensor, fused_tensor = predict_fusion_with_separate_models(
            ir_encoder_path, vi_encoder_path, decoder_path, fusion_model_path,
            ir_image_path, vi_image_path, output_path)
        print("红外与可见光图像融合测试完成!")

        # 检查输入图像是否为彩色图像
        ir_img = Image.open(ir_image_path)
        vi_img = Image.open(vi_image_path)
        is_color = ir_img.mode == 'RGB' and vi_img.mode == 'RGB'

        if is_color:
            print("\n检测到彩色图像，使用彩色处理模式...")
            print("使用分离的编码器和解码器模型进行红外与可见光彩色图像融合...")
            ir_rgb, vi_rgb, fused_rgb = predict_color_fusion_with_separate_models(
                ir_encoder_path, vi_encoder_path, decoder_path, fusion_model_path,
                ir_image_path, vi_image_path, color_output_path)
            print("红外与可见光彩色图像融合测试完成!")
    except Exception as e:
        print(f"测试过程中出现错误: {e}")


使用分离的编码器和解码器模型进行红外与可见光图像融合...
使用设备: cuda
已加载红外编码器权重
已加载可见光编码器权重
已加载解码器权重
红外图像尺寸: torch.Size([1, 1, 1024, 1280])
可见光图像尺寸: torch.Size([1, 1, 1024, 1280])
融合图像尺寸: torch.Size([1, 1, 1024, 1280])


C:\Users\DELL\AppData\Local\Temp\ipykernel_8332\4187320360.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.ir_encoder.load_state_dict(torch.load(ir_encoder_path, 

融合对比图像已保存到: ../output/fusion/fusion_comparison.png
红外与可见光图像融合测试完成!

检测到彩色图像，使用彩色处理模式...
使用分离的编码器和解码器模型进行红外与可见光彩色图像融合...
使用设备: cuda
已加载红外编码器权重
已加载可见光编码器权重
已加载解码器权重
红外图像尺寸: torch.Size([1, 1, 1024, 1280])
可见光图像尺寸: torch.Size([1, 1, 1024, 1280])
融合图像尺寸: torch.Size([1, 1, 1024, 1280])
彩色融合对比图像已保存到: ../output/fusion/color_fusion_comparison.png
重建彩色图像已保存到: ../output/fusion\color_fusion_comparison_color_fused.png
红外与可见光彩色图像融合测试完成!
